# Cover Letter Agent - V2, written paragraph by paragraph

Two paragraphs are drafted independently and in parallel, then reconciled into one letter - plus
a closing paragraph written from scratch - by a single review pass at the end.

```
               ┌─> prepare ──────────────────────────────────────────┐
START ─> load ─┼─> research ──> select_reasons ──> para_intro ───────┼─> assemble ─> END
               └─> qualify ──> select_qualifications ──> para_body ──┘
```

- `prepare` pulls closing paragraphs from your past letters.
- `research` finds facts about the company/team, each tied to a source URL, plus candidate
  reasons for wanting the role.
- `qualify` identifies which of the candidate's qualifications matter most and what each is worth
  to the team - it fires directly off `load`, alongside `prepare`/`research`, since it doesn't
  need anything from `prepare` any more; that lets its LLM call overlap with `research`'s slow web
  searches instead of only starting once `prepare` (and, since they share a tick, `research` too)
  has already finished.
- `select_reasons` pauses for a human to pick which reasons open the letter (or write their own)
  when `select_reasons_in_the_loop` is set; `select_qualifications` does the same for which
  qualifications the body builds on, independently, when `select_qualifications_in_the_loop` is
  set - each auto-picks the top-ranked ones otherwise. See the notes in sections 7 and 8.
- `para_intro` and `para_body` each depend only on their own upstream branch - neither sees the
  other's output - so both run concurrently.
- `assemble` stitches the two paragraphs, runs deterministic checks (placeholders, salutation),
  then a single LLM pass writes the closing paragraph (from `prepare`'s past-letter closings) and
  fixes everything found, plus any duplication or disjointedness left over from writing `para_intro`
  and `para_body` blind to each other - one call doing both jobs, rather than a dedicated node for
  the closing.

## Why it is shaped this way

**Each node emits a human-legible artifact.** When paragraph two is weak you know which node to
fix. That is not true of designs whose intermediate products are evidence maps and briefs.

**`qualify` asks the question most cover letters get wrong.** Not "which of my qualifications are
relevant" but *how each one would add value to this team*. A letter saying "I have X" is weaker
than one saying "X solves Y for you", and this is the node that forces the second form.

**Paragraphs are written blind to each other, on purpose, for latency.** `research` is the slow
step (multiple web searches); running `qualify`/`para_body` concurrently with it hides their
latency instead of adding to it. The trade-off - paragraphs that might repeat a hook or read
disjointedly - is caught by `assemble`'s review pass instead of prevented by sequencing.

**The closing is written inside `assemble`, not its own node.** `assemble` already needs to read
both paragraphs in full to reconcile them, so writing the closing there is a free extension of
that same call rather than a third parallel branch and a dedicated LLM call for a short, formulaic
paragraph.

**One review pass at the end, not a fact-check loop.** `assemble` no longer checks claims against
the CV; it only reconciles the independently-written paragraphs into one coherent letter. The V1
experiment showed two gates arguing with each other - a quality reviewer demanding stronger claims
while a fact checker rejected them - so this stays a single pass, not a loop.


---
## 1. Setup

```bash
uv venv --python 3.13
uv pip install langgraph langchain-anthropic python-dotenv ipykernel pypdf python-docx
```

`.env` next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

In [ ]:
import os
from dotenv import find_dotenv, load_dotenv

# must run before the langchain/langgraph imports below: langsmith caches whether tracing is
# enabled the first time anything reads the env var, so importing those packages first makes
# LANGSMITH_TRACING silently no-op even once .env is loaded
load_dotenv(find_dotenv(usecwd=True))

import html
import inspect
import json
import re
import uuid
from dataclasses import dataclass, asdict
from datetime import date
from pathlib import Path
from typing import Optional, TypedDict

from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

assert os.environ.get("ANTHROPIC_API_KEY"), "Put ANTHROPIC_API_KEY in a .env file next to this notebook"
print("Anthropic key loaded.")


---
## 2. Configuration

The word budgets are per paragraph rather than for the letter as a whole. Three writers each
asked for "a paragraph" reliably sprawl; given a number, they do not.

In [ ]:
@dataclass
class Config:
    model: str = "claude-opus-5"
    output_language: str = "English"

    # per-section word budgets
    intro_words: int = 90
    body_words: int = 190
    close_words: int = 45
    max_words: int = 400        # hard ceiling for the assembled letter

    web_search_max_uses: int = 6
    dump_prompts: bool = True   # write every rendered prompt to outputs/.prompts/

    # evaluate/revise loop: a score at or above this passes; below it triggers exactly one
    # revise pass (bounded by State's revision_count, not by this) before returning regardless
    eval_score_threshold: int = 4

    inputs_dir: Path = Path("inputs")
    outputs_dir: Path = Path("outputs")


cfg = Config()
cfg.outputs_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps({k: str(v) for k, v in asdict(cfg).items()}, indent=2))

---
## 3. Inputs

```
inputs/
├── job_url.txt         # optional - a link to the posting
├── job_posting.txt     # used when there is no URL, or when fetching it fails
├── cv.txt              # or .pdf / .docx
└── past_letters/       # any number of letters you wrote before
```

If `job_url.txt` holds a link, Claude fetches it. **Expect that to fail often** - LinkedIn,
Workday and Greenhouse postings are JavaScript-rendered or behind a login, so a fetch returns a
shell page or a sign-in wall rather than the advertisement. The loader checks for that and falls
back to the pasted text instead of quietly building a letter from an empty page.

In [ ]:
def read_document(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {".txt", ".md"}:
        return path.read_text(encoding="utf-8")
    if suffix == ".pdf":
        from pypdf import PdfReader
        return "\n".join((page.extract_text() or "") for page in PdfReader(str(path)).pages)
    if suffix == ".docx":
        import docx
        return "\n".join(p.text for p in docx.Document(str(path)).paragraphs)
    raise ValueError(f"Unsupported file type: {path.name}")


def find(stem: str) -> Optional[Path]:
    matches = [p for p in cfg.inputs_dir.glob(f"{stem}.*")
               if p.suffix.lower() in {".txt", ".md", ".pdf", ".docx"}]
    return matches[0] if matches else None


LOGIN_WALL = re.compile(r"(sign in|log in|create an account|enable javascript|captcha|"
                        r"access denied|403 forbidden)", re.I)


def fetch_job_from_url(url: str) -> Optional[str]:
    """Fetch a posting with Claude's web_fetch tool. Returns None when the page is unusable."""
    fetcher = ChatAnthropic(model=cfg.model, max_tokens=8000).bind_tools(
        [{"type": "web_fetch_20260209", "name": "web_fetch", "max_uses": 2}])
    try:
        message = fetcher.invoke([HumanMessage(content=(
            f"Fetch {url} and reproduce the job advertisement it contains, verbatim and in full: "
            f"title, responsibilities and requirements. If the page does not contain a job "
            f"advertisement - a login wall, an error, or an empty shell - reply with exactly "
            f"NO_POSTING_FOUND and nothing else."))])
    except Exception as exc:
        print(f"[inputs] fetch failed: {type(exc).__name__}")
        return None

    text = message.text if isinstance(message.text, str) else message.text()
    if "NO_POSTING_FOUND" in text or len(text.split()) < 150 or LOGIN_WALL.search(text[:600]):
        print("[inputs] the URL did not yield a usable posting")
        return None
    return text


def load_inputs() -> dict:
    posting = None
    url_file = cfg.inputs_dir / "job_url.txt"
    if url_file.exists() and url_file.read_text().strip().startswith("http"):
        url = url_file.read_text().strip()
        print(f"[inputs] fetching {url}")
        posting = fetch_job_from_url(url)

    if posting is None:
        path = find("job_posting")
        if path is None:
            raise FileNotFoundError("No usable job_url.txt and no job_posting.(txt|md|pdf|docx)")
        posting = read_document(path)
        print(f"[inputs] using {path.name}")

    letters_dir = cfg.inputs_dir / "past_letters"
    letter_paths = sorted(p for p in letters_dir.glob("*")
                          if p.suffix.lower() in {".txt", ".md", ".pdf", ".docx"}) if letters_dir.exists() else []

    data = {"job_ad": posting,
            "cv": read_document(find("cv")),
            "past_letters": [read_document(p) for p in letter_paths]}

    print(f"job ad  : {len(data['job_ad'].split())} words")
    print(f"cv      : {len(data['cv'].split())} words")
    print(f"letters : {len(letter_paths)}")
    return data


inputs = load_inputs()  # SHIM:SKIP - notebook-only; the `load` node does this at run time


---
## 4. Schemas, state and helpers

The state is plain strings and lists. Only the qualifications are typed, because that's the one
thing Python itself reads back apart from text. Everything else is text, since its only consumer
is the next prompt.

In [ ]:
class Qualification(BaseModel):
    qualification: str = Field(description="education, experience, skill or knowledge")
    evidence: str = Field(description="the specific thing on the CV that establishes it")
    value_to_team: str = Field(description="what it would let this particular team do, or do "
                                           "better - phrased from their side, not the candidate's")
    relevance: int = Field(ge=1, le=5)


class Qualifications(BaseModel):
    items: list[Qualification]


class ResearchSummary(BaseModel):
    company: str = Field(description="the hiring company's name, exactly as it appears")
    role: str = Field(description="the role's title, exactly as it appears")
    reasons: list[str] = Field(description="the candidate motivations/reasons listed, exactly as "
                                           "given, most probable first - do not summarize, "
                                           "reword or invent any")


class Assembled(BaseModel):
    letter: str = Field(description="the final cover letter - the given opening and body, plus "
                                    "the closing paragraph you write, with every listed problem "
                                    "fixed and duplication/coherence issues resolved")
    changes: list[str] = Field(description="a short list of what was changed and why, for a "
                                           "human reviewing the output")


class LetterEvaluation(BaseModel):
    grounded: bool = Field(description="every claim about the candidate is supported by the CV "
                                       "or the candidate's past cover letters")
    unsupported_claims: list[str] = Field(description="claims that aren't grounded, quoted")
    specific: bool = Field(description="clearly about this role and company - if you could swap "
                                       "in a competitor's name and it would still read sensibly, "
                                       "this fails")
    coherent: bool = Field(description="the three paragraphs don't repeat the same hook, fact or "
                                       "phrase - they were written independently of each other "
                                       "and may have converged on the same point")
    flows_well: bool = Field(description="no sudden topic changes - each paragraph and each "
                                         "sentence follows naturally from what came before, "
                                         "rather than reading as unconnected blocks")
    issues: list[str] = Field(description="concrete problems to fix, specific enough that "
                                          "someone revising the letter would know exactly what "
                                          "to change")
    overall_score: int = Field(ge=1, le=5, description="1 = needs major rework, 5 = no changes "
                                                        "needed")


class RevisedLetter(BaseModel):
    letter: str = Field(description="the corrected cover letter")
    changes: list[str] = Field(description="what was changed and why")


class State(TypedDict, total=False):
    job_ad: str
    cv: str
    past_letters: list

    closings: list

    research_notes: str
    sources: list

    # candidate_reasons: research()'s ranked list, offered as a menu by select_reasons().
    # selected_reasons: what the human actually picked there (plus anything they typed
    # themselves) - this, not candidate_reasons, is what para_intro writes from.
    candidate_reasons: list
    selected_reasons: list

    # qualifications: qualify()'s ranked list, offered as a menu by select_qualifications().
    # selected_qualifications: what was picked there (plus any custom entries) - this, not
    # qualifications, is what para_body writes from.
    qualifications: list
    selected_qualifications: list
    para_intro: str
    para_body: str

    final_letter: str
    changes: list

    evaluation_issues: list
    evaluation_unsupported_claims: list
    evaluation_score: int
    revision_count: int

    # set by the caller - each gates one node independently, so any combination of the three can
    # be interactive while the rest run automatically
    select_reasons_in_the_loop: bool          # select_reasons
    select_qualifications_in_the_loop: bool   # select_qualifications
    human_in_the_loop: bool                   # human_review
    human_action: str

    # set by research() (see its docstring for why) - used by assemble()/revise() via
    # save_letter() to name the output file. Not otherwise exposed: every other node still reads
    # job_ad directly rather than these pre-extracted copies of the same information
    company: str
    role: str


def log_prompt(system: str, user: str, node: str = None) -> None:
    """Write the fully rendered prompt to outputs/.prompts/<node>.md.

    The node name is taken from the call stack by default - log_prompt <- ask/prose <- the node -
    so no call site has to pass it, and a new node gets prompt logging for free. Pass `node`
    explicitly when logging from the node function itself rather than through ask/prose."""
    if not cfg.dump_prompts:
        return
    node = node or inspect.stack()[2].function
    directory = cfg.outputs_dir / ".prompts"
    directory.mkdir(parents=True, exist_ok=True)
    (directory / f"{node}.md").write_text(
        f"# {node}\n\n## SYSTEM\n\n{system}\n\n## USER\n\n{user}\n", encoding="utf-8")


def ask(schema, system: str, user: str, attempts: int = 3, max_tokens: int = 16000):
    """A structured call, retried on a malformed response.

    Keep max_tokens generous - Claude Opus 5 thinks by default and those tokens come out of the
    same budget, so starving it truncates the tool call rather than the prose."""
    log_prompt(system, user)
    chain = ChatAnthropic(model=cfg.model, max_tokens=max_tokens).with_structured_output(schema)
    messages = [SystemMessage(content=system), HumanMessage(content=user)]
    for attempt in range(1, attempts + 1):
        try:
            return chain.invoke(messages)
        except Exception as exc:
            if attempt == attempts:
                raise
            print(f"    ({schema.__name__} attempt {attempt} failed: {type(exc).__name__}, retrying)")


def prose(system: str, user: str, max_tokens: int = 12000) -> str:
    log_prompt(system, user)
    message = ChatAnthropic(model=cfg.model, max_tokens=max_tokens).invoke(
        [SystemMessage(content=system), HumanMessage(content=user)])
    text = message.text if isinstance(message.text, str) else message.text()
    if not text.strip():
        raise RuntimeError("Model returned no text - thinking consumed the budget; raise max_tokens.")
    return text.strip()


STYLE_RULES = """- Language: {language}.
- Output the paragraph text only. No heading, no preamble, no commentary."""


def style() -> str:
    return STYLE_RULES.format(language=cfg.output_language)


print("schemas, state and helpers defined")


---
## 5. Node: `prepare`

Pure Python, no model call: split each past letter into paragraphs, drop the sign-off, keep the
last real paragraph. Those become the models `assemble` writes the new closing from.

In [ ]:
SIGNOFF = re.compile(r"^(kind regards|yours sincerely|yours faithfully|best regards|sincerely|"
                     r"regards|many thanks|thank you,)", re.I | re.M)


def extract_closing(letter: str) -> Optional[str]:
    """The last real paragraph before the sign-off."""
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", letter) if p.strip()]
    while paragraphs and (SIGNOFF.match(paragraphs[-1]) or len(paragraphs[-1].split()) <= 5):
        paragraphs.pop()
    return paragraphs[-1] if paragraphs else None


def prepare(state: State) -> dict:
    """Pulls a model closing paragraph out of each past letter, for assemble() to write the new
    closing from - pure Python, no model call needed to find the end of a letter."""
    letters = state.get("past_letters", [])
    closings = [c for c in (extract_closing(letter) for letter in letters) if c]
    print(f"[prepare] {len(closings)} closing paragraph(s) extracted")
    return {"closings": closings}


---
## 6. Node: `research`

Searches the company **and the team this role sits in**, then turns the notes into facts that each
carry a URL. Python drops any fact whose URL was never visited.

Also extracts the hiring company's name and role title from the job advertisement, since this node
already has to read it closely to know who to search for - `assemble()`/`revise()` reuse these
instead of reading the job ad a second time just to name the output file.

Researching the team rather than just the company is what makes the opening hard for another
applicant to replicate. "Coolblue is a large Dutch retailer" is available to everyone; how its
delivery and installation operation actually works is not.

The second output is reasons - why a candidate might genuinely want this particular company or
team. It draws on both the research findings and what the job advertisement itself says - research
never sees the CV, so this is not personalised against it.

In [ ]:
RESEARCH_PROMPT = """Read this job advertisement, identify the hiring company and the team the role sits in (if mentioned in the job advertisement), then research both using the web_search tool.

In light of your research and the information given in the job advertisement, find specific, checkable facts rather than marketing adjectives:
1. What the company does, its scale and market position.
2. How the team's part of the business actually works day to day - its operations, its named
   services, the problems it owns.
3. Concrete recent developments: expansion, technology investment, published engineering work.
   Prefer the last two years.
4. Documented engineering or data practice - tech blogs, conference talks, reports.
5. Stated culture and working practices, in the company's own words where possible.

Call the web_search tool directly for each query. Do not invoke it indirectly through code
execution or any scripting tool - that path is not supported and will error. If a direct
web_search call itself errors, only that specific call failed; before concluding search is
unavailable, check back through every web_search result you already received in this same
response - do not contradict or discard results you already have when writing your answer.

JOB ADVERTISEMENT:
{ad}

Once you are done researching, write your answer as four sections, in this order:

COMPANY: the hiring company's name, exactly as the job advertisement gives it.

ROLE: the role's title, exactly as the job advertisement gives it.

FINDINGS:
- one checkable fact per line, each followed by its source URL in square brackets, e.g. "runs
  its own delivery fleet and installs appliances in the home [https://source.url]". Use only
  URLs you actually retrieved via web_search in this response - discard anything you cannot
  attribute to one of them, including things you happen to know independently. A fact is never
  an adjective: "runs its own delivery fleet" is a fact; "is an innovative company" is not.

REASONS:
- why I might genuinely want to work at this specific company or in this specific team (my
  motivations), considering BOTH the findings above AND what the job advertisement itself says
  about the company and the team. Rank them, most probable first.
"""


EXTRACT_SUMMARY_SYSTEM = """Extract three things from this research answer about a job
advertisement's hiring company and team:

- `company`: the hiring company's name, exactly as it appears.
- `role`: the role's title, exactly as it appears.
- `reasons`: the candidate motivations/reasons listed, as a list of strings, most probable first,
  exactly as given - do not summarize, reword, merge or invent any."""

def research(state: State) -> dict:
    """Search and ground facts in one call, returning them as a single text field, plus the
    hiring company, role and candidate reasons. Extracted via a small structured-output call
    (below) rather than by regex on the model's own COMPANY:/ROLE:/REASONS: heading formatting -
    a regex silently comes back empty if the model doesn't format a heading exactly as asked
    (different casing, markdown emphasis, no trailing blank line), and since this feeds a
    human-facing picker (select_reasons) it needs to actually be reliable, not just usually
    right. ask()'s built-in retry-on-malformed-response covers the rest."""
    searcher = ChatAnthropic(model=cfg.model, max_tokens=16000).bind_tools(
        [{"type": "web_search_20260209", "name": "web_search", "max_uses": cfg.web_search_max_uses}])
    prompt = RESEARCH_PROMPT.format(ad=state["job_ad"])
    log_prompt("(single combined message - no separate system prompt)", prompt, node="research")

    result = searcher.invoke([HumanMessage(content=prompt)])
    text = result.text if isinstance(result.text, str) else result.text()
    if not text.strip():
        raise RuntimeError("research returned no text - thinking consumed the budget; raise max_tokens.")

    summary = ask(ResearchSummary, EXTRACT_SUMMARY_SYSTEM, f"RESEARCH ANSWER:\n{text}")
    company, role, candidate_reasons = summary.company, summary.role, list(summary.reasons)

    # best-effort cosmetic trim of the COMPANY:/ROLE:/REASONS: headings out of research_notes -
    # the actual data is already captured above via the structured call, so this not matching
    # exactly just leaves those lines sitting in research_notes harmlessly, not a functional bug
    text = re.sub(r"^(COMPANY|ROLE):.*\n?", "", text, flags=re.M | re.I)
    text = re.split(r"^\s*#{0,3}\s*\**REASONS\**:?\s*$", text, maxsplit=1, flags=re.M | re.I)[0]

    sources, seen = [], set()
    for block in result.content:
        if not isinstance(block, dict) or block.get("type") != "web_search_tool_result":
            continue
        content = block.get("content")
        if not isinstance(content, list):          # an error object rather than results
            print(f"[research] search error: {content}")
            continue
        for item in content:
            if item.get("url") and item["url"] not in seen:
                seen.add(item["url"])
                sources.append({"title": item.get("title", ""), "url": item["url"]})

    # drop any line citing a URL that was never actually retrieved - this check is agnostic to
    # which section (FINDINGS vs REASONS) a line belongs to, since it only fires on lines that
    # cite a URL at all; REASONS lines never do, so they pass through untouched
    kept, dropped = [], 0
    for line in text.splitlines():
        cited = re.findall(r"https?://[^\s\]]+", line)
        if cited and not any(url in seen for url in cited):
            dropped += 1
            continue
        kept.append(line)

    notes = html.unescape("\n".join(kept)).strip()

    print(f"[research] {company!r} / {role!r} - {len(sources)} sources, "
          f"{len(candidate_reasons)} candidate reasons, {dropped} ungrounded line(s) dropped")

    return {"company": company, "role": role, "research_notes": notes,
            "candidate_reasons": candidate_reasons, "sources": sources}


---
## 7. Nodes: `select_reasons` and `para_intro`

**"Why I want to work for this company and this team."**

`select_reasons` pauses, when `select_reasons_in_the_loop` is set, so a human picks up to 3 of
`research()`'s candidate reasons for wanting the role, or writes their own (otherwise it falls
back to the top 2 by rank). `para_intro` then writes the opening paragraph built on exactly those,
using the research notes only as background for showing understanding of what the company/team
actually does, not as a source of reasons to state.

Deciding *why you want this role* is something worth putting in a human's hands rather than
leaving to the model to guess at - this flag exists so that choice is yours when you want it.

In [ ]:
def select_reasons(state: State) -> dict:
    """Pauses so a human picks which of research()'s candidate reasons para_intro should build
    the opening paragraph on, and optionally adds their own. Only when
    select_reasons_in_the_loop is set (independent of select_qualifications_in_the_loop - see
    State); otherwise falls back to the top 2 by rank, since research() already ranks them
    most-probable-first.

    Resume with Command(resume=...) where the payload is {"selected": [0, 2], "custom": ["..."]}
    - `selected` are indices into the `reasons` list the interrupt showed, capped at 3; `custom`
    is any number of the human's own reasons, added on top."""
    reasons = state.get("candidate_reasons", [])
    if not state.get("select_reasons_in_the_loop"):
        return {"selected_reasons": reasons[:2]}

    print("[select_reasons] waiting for a selection...")
    decision = interrupt({
        "gate": "select_reasons",
        "reasons": reasons,
    }) or {}

    picked = [reasons[i] for i in decision.get("selected", [])[:3] if 0 <= i < len(reasons)]
    custom = [r.strip() for r in decision.get("custom", []) if r and r.strip()]

    print(f"[select_reasons] {len(picked)} selected, {len(custom)} custom")
    return {"selected_reasons": picked + custom}


INTRO_SYSTEM = """Write the opening paragraph of my cover letter for the below job advertisement,
grounded in the research notes about the company/team and built on my selected
reasons for wanting this role. Its single job is to answer why I want to work for
this company and this team/role (my motivation).

- Around {words} words. One paragraph.
- Build the paragraph on my selected reasons given below - these were chosen by me, so use all of them if they fit naturally within the word budget,
  otherwise prioritise the ones that fit best together.
- Do not use long sentences.
- Show understanding of what the company/team actually does.
- Begin with a salutation on its own line ("Dear Hiring Team," unless the advertisement names
  someone), then a blank line, then the paragraph.

""" + "{style}"


def para_intro(state: State) -> dict:
    reasons = "\n".join(f"- {r}" for r in state.get("selected_reasons", [])) or "(none selected)"
    text = prose(
        INTRO_SYSTEM.format(words=cfg.intro_words, style=style()),
        f"JOB ADVERTISEMENT:\n{state['job_ad']}\n\n"
        f"RESEARCH NOTES (background on the company/team, not reasons to state directly):\n"
        f"{state['research_notes']}\n\n"
        f"CANDIDATE'S SELECTED REASONS (build the paragraph on these):\n{reasons}")

    print(f"[para_intro] {len(text.split())} words")
    return {"para_intro": text}

---
## 8. Nodes: `qualify` and `select_qualifications`

The analytical step behind paragraph two, and the one that most improves the letter.

For every qualification `qualify` records not just the evidence but **`value_to_team`** - what
this would let *that team* do, or do better, phrased from their side. The difference between "I
have built multi-output forecasters" and "their roster planning needs one forecast per slot per
day, which is the same shape" is the difference between a letter about you and a letter about
them.

`select_qualifications` then pauses, when `select_qualifications_in_the_loop` is set (independent
of `select_reasons_in_the_loop` - either, both or neither can be on), so a human picks which of
these `para_body` should actually build on, and optionally adds one of their own - no cap, unlike
`select_reasons`, since `para_body`'s own prompt already limits itself to building on two or three
properly. Off, it falls back to the top 6 by relevance.

In [ ]:
QUALIFY_SYSTEM = """Using my CV and past cover letters, identify the qualifications that matter most for this job advertisement, and explain the specific value each qualification would bring to this team.

For each entry:
- `qualification`: education, experience, skill or knowledge. It needs to be a full sentence.
- `evidence`: the specific thing in the CV or past letters that establishes it. Quote or closely
  paraphrase. If you cannot point to something concrete, the qualification does not belong here.
- `value_to_team`: what this qualification would let THIS team do, or do better. Write it from their side, in
  terms of their problems - not as a restatement of the my experience. This is the most
  important field; a generic benefit that would apply to any team means the entry is weak.
- `relevance`: 1-5 against what the advertisement actually emphasises.

Return eight entries, strongest first."""


def qualify(state: State) -> dict:
    letters = "\n\n--- letter ---\n\n".join(state.get("past_letters", [])[:3]) or "(none)"
    result = ask(Qualifications, QUALIFY_SYSTEM,
                 f"JOB ADVERTISEMENT:\n{state['job_ad']}\n\n"
                 f"CV:\n{state['cv']}\n\n"
                 f"PAST COVER LETTERS (for additional detail about the candidate's work):\n{letters}")

    items = sorted((q.model_dump() for q in result.items), key=lambda q: -q["relevance"])
    print(f"[qualify] {len(items)} qualifications")
    for item in items[:4]:
        print(f"    [{item['relevance']}] {item['qualification']}")
        print(f"        value: {item['value_to_team'][:95]}")
    return {"qualifications": items}


def select_qualifications(state: State) -> dict:
    """Pauses so a human picks which of qualify()'s ranked qualifications para_body should build
    the body paragraph on, and optionally adds their own (qualification + value_to_team only - no
    evidence field, since that's meant to point at something in the CV; a custom entry gets
    marked as candidate-supplied instead, and evaluate()'s grounded check is the safety net if it
    turns out unsupported). No selection cap - para_body's own prompt already says to build on
    two or three properly rather than listing everything.

    Only when select_qualifications_in_the_loop is set (independent of
    select_reasons_in_the_loop - see State); otherwise falls back to the top 6 by relevance,
    exactly what para_body used to take directly before this gate existed.

    Resume with Command(resume=...) where the payload is {"selected": [0, 2],
    "custom": [{"qualification": "...", "value_to_team": "..."}]} - `selected` are indices into
    the `qualifications` list the interrupt showed."""
    qualifications = state.get("qualifications", [])
    if not state.get("select_qualifications_in_the_loop"):
        return {"selected_qualifications": qualifications[:6]}

    print("[select_qualifications] waiting for a selection...")
    decision = interrupt({
        "gate": "select_qualifications",
        "qualifications": qualifications,
    }) or {}

    picked = [qualifications[i] for i in decision.get("selected", []) if 0 <= i < len(qualifications)]
    custom = []
    for entry in decision.get("custom", []):
        qualification = (entry.get("qualification") or "").strip()
        value_to_team = (entry.get("value_to_team") or "").strip()
        if qualification and value_to_team:
            custom.append({"qualification": qualification,
                            "evidence": "(candidate-supplied, not drawn from the CV)",
                            "value_to_team": value_to_team, "relevance": 5})

    print(f"[select_qualifications] {len(picked)} selected, {len(custom)} custom")
    return {"selected_qualifications": picked + custom}

---
## 9. Node: `para_body`

**"Why I am a good fit for this role."**

Written independently of `para_intro`, in parallel with it - see the top of the notebook for why.
Any overlap between the two gets caught and fixed later, in `assemble`.

Two or three paragraphs rather than one. A single paragraph carrying education, experience,
skills and knowledge is either enormous or shallow.

In [ ]:
BODY_SYSTEM = """Write the body of my cover letter: why I am a good fit for this role.

- Two or three paragraphs, {words} words in total.
- Use the qualifications supplied, and the evidence given with them. Invent no metric,
  date, tool or responsibility. evidence field shows the specific thing in the CV or past cover letters that establishes the qualification.
- Mention something I actually did, then connect it to what the team
  needs - the `value to team` line tells you what that connection is. The point of the paragraph
  is what the team gets, not what I have.


""" + "{style}"


def para_body(state: State) -> dict:
    chosen = state.get("selected_qualifications", [])
    rendered = "\n\n".join(
        f"[{q['relevance']}/5] {q['qualification']}\n"
        f"    evidence: {q['evidence']}\n"
        f"    value to team: {q['value_to_team']}" for q in chosen)

    text = prose(
        BODY_SYSTEM.format(words=cfg.body_words, style=style()),
        f"JOB ADVERTISEMENT:\n{state['job_ad']}\n\n"
        f"QUALIFICATIONS:\n{rendered}")

    print(f"[para_body] {len(text.split())} words")
    return {"para_body": text}


---
## 10. Node: `assemble`

Stitching, checking, writing the closing, and one review pass - always run, not conditional.

The deterministic checks come first - placeholders, salutation - because those are facts about a
string and do not need a model. They're handed to the review call as a fixed list of problems to
resolve. Length and sign-off aren't checked here, only afterwards: the closing (which decides
both) doesn't exist yet at this point - it's written by the same call these problems feed into.

Since `para_intro` and `para_body` were written independently (see the top of the notebook), this
is also where any repeated hook, fact or phrasing across them gets caught and reconciled - the
review pass has explicit license to lightly edit for flow and coherence, but not to invent new
claims. It also writes the closing paragraph itself, modelled on `prepare`'s past-letter closings -
one call doing both jobs, since it already has to read both paragraphs in full to reconcile them,
rather than a dedicated node and LLM call for a short, formulaic paragraph. It does not check the
letter's claims against the CV; the CV is background context only, for editing accurately.

Deliberately one pass, not a loop: the V1 experiment showed two gates arguing with each other, a
quality reviewer demanding stronger claims while a fact checker rejected them, converging on
nothing. The same call reports what it changed and why. It reuses the company/role `research()`
already extracted (see section 6) to name the output file, rather than extracting them again here.


In [ ]:
REVISE_SYSTEM = """You are given my cover letter's opening and body paragraphs - written
independently of each other - my own past cover letters' closings (for style), my CV for context, and a list of problems found by deterministic checks.

Write the closing paragraph, then produce the final, corrected letter:
- The closing: around {close_words} words, short, forward-looking, no new claims about my experience or qualifications. Match the structure and register of my
  own past closings, given below - their length, their level of formality, how they make the ask
  - without copying their sentences word for word; this is a different application. End with a
  sign-off line matching the one the past letters use ("Kind regards," or similar) on its own
  line, then my name on the line after it. Both are required.
- Keep the total letter, opening and body included, to at most {max_words} words.
- Fix every problem listed.
- Because the opening and body were written without seeing each other, they may repeat a hook,
  fact or phrase, or read disjointedly at the boundary. Find and fix this: cut or merge repeated
  points, smooth transitions, and keep the tone consistent throughout, into the closing you write.
- You may lightly edit wording for flow and coherence. Do not invent new claims, facts, metrics
  or responsibilities that are not already present in the letter.
- The CV is background context only, to help you edit accurately and consistently - you are not
  checking the letter's claims against it or removing anything for lack of CV support.
- Language: {language}.

Report a short list of what you changed and why."""

PLACEHOLDER_RE = re.compile(r"(\[[A-Za-z][^\]]{0,40}\]|\{\{.*?\}\}|\bTODO\b|\bXXXX?\b)")


def save_letter(company: str, role: str, letter: str) -> Path:
    """Shared by assemble() and revise() - revise() overwrites the same file assemble() already
    wrote, under the same name, since a revision doesn't change the company/role it's filed
    under."""
    slug = re.sub(r"[^a-z0-9]+", "-", f"{company}-{role}".lower()).strip("-")[:60]
    path = cfg.outputs_dir / f"{date.today().isoformat()}_{slug}_v2.md"
    path.write_text(letter, encoding="utf-8")
    return path


def assemble(state: State) -> dict:
    intro_and_body = f"{state['para_intro']}\n\n{state['para_body']}".strip()
    intro_and_body = re.sub(r"\n{3,}", "\n\n", intro_and_body)

    def deterministic(text: str) -> list:
        # LENGTH and SIGN-OFF aren't checked here - the closing (which decides both) doesn't
        # exist yet at this point, it's written by the same call these problems feed into. They're
        # checked afterwards instead, against revised.letter - see below.
        problems = []
        for match in set(PLACEHOLDER_RE.findall(text)):
            problems.append(f"PLACEHOLDER: unfilled {match!r}")
        if not re.match(r"^(dear|to whom)", text.strip(), re.I):
            problems.append("SALUTATION: the letter does not open with one")
        return problems

    problems = deterministic(intro_and_body)
    print(f"[assemble] {len(intro_and_body.split())} words (opening+body), {len(problems)} problem(s)")
    for problem in problems:
        print(f"    {problem}")

    examples = "\n\n--- past closing ---\n\n".join(state.get("closings", [])) or "(none supplied)"

    revised = ask(Assembled, REVISE_SYSTEM.format(close_words=cfg.close_words, max_words=cfg.max_words,
                                                   language=cfg.output_language),
                  f"CANDIDATE'S CV (background context only):\n{state['cv']}\n\n"
                  f"THE CANDIDATE'S OWN PAST CLOSINGS, to model the new closing on:\n{examples}\n\n"
                  f"PROBLEMS FOUND:\n" + ("\n".join(f"- {p}" for p in problems) or "(none)") + "\n\n"
                  f"LETTER SO FAR (para_intro and para_body, written independently - write the "
                  f"closing paragraph to follow them):\n"
                  f"---\n{intro_and_body}\n---")

    print(f"[assemble] revised -> {len(revised.letter.split())} words")
    for change in revised.changes:
        print(f"    - {change}")

    words = len(revised.letter.split())
    if words > cfg.max_words:
        print(f"    ! LENGTH: {words} words, limit {cfg.max_words} (post-check, not auto-fixed)")
    if not SIGNOFF.search("\n".join(revised.letter.strip().splitlines()[-3:])):
        print("    ! SIGN-OFF: no sign-off line before the name (post-check, not auto-fixed)")

    path = save_letter(state["company"], state["role"], revised.letter)
    print(f"[assemble] -> {path}")

    return {"final_letter": revised.letter, "changes": revised.changes}


EVALUATE_SYSTEM = """Judge a finished cover letter against the candidate's CV and their past
cover letters (both are background material the letter's claims should be traceable to - a claim
grounded in a past letter is as valid as one grounded in the CV) and against the job
advertisement.

Check:
- `grounded`: every claim about the candidate is supported by the CV or the past letters. List
  anything that isn't in `unsupported_claims`, quoted.
- `specific`: the letter is clearly about this role and company - if you could swap in a
  competitor's name and it would still read sensibly, it fails this check.
- `coherent`: the three paragraphs don't repeat the same hook, fact or phrase - they were written
  independently of each other and may have converged on the same opening move or point.
- `flows_well`: no sudden topic changes - each paragraph, and each sentence within it, should
  follow naturally from what came before. A letter that reads as three unconnected blocks stapled
  together fails this even if each block is individually fine.

List concrete problems in `issues` - specific enough that someone fixing the letter would know
exactly what to change. Score `overall_score` 1-5, where 5 means no changes needed."""


def evaluate(state: State) -> dict:
    letters = "\n\n--- past letter ---\n\n".join(state.get("past_letters", [])) or "(none supplied)"
    result = ask(LetterEvaluation, EVALUATE_SYSTEM,
                 f"JOB ADVERTISEMENT:\n{state['job_ad']}\n\n"
                 f"CV:\n{state['cv']}\n\n"
                 f"CANDIDATE'S PAST COVER LETTERS:\n{letters}\n\n"
                 f"LETTER TO JUDGE:\n---\n{state['final_letter']}\n---")

    print(f"[evaluate] score {result.overall_score}/5 - grounded={result.grounded} "
          f"specific={result.specific} coherent={result.coherent} flows_well={result.flows_well}")
    for issue in result.issues:
        print(f"    - {issue}")

    return {"evaluation_issues": result.issues,
            "evaluation_unsupported_claims": result.unsupported_claims,
            "evaluation_score": result.overall_score}


REVISE_LOOP_SYSTEM = """You are given a cover letter, an evaluator's findings about it, the
candidate's CV and past cover letters for grounding, and the job advertisement.

Fix exactly what the evaluator flagged - the unsupported claims and the listed issues. Do not
invent new claims, facts, metrics or responsibilities. You may lightly edit wording where that's
needed for flow or coherence, but leave everything else as it is.

Report a short list of what you changed and why."""


def revise(state: State) -> dict:
    letters = "\n\n--- past letter ---\n\n".join(state.get("past_letters", [])) or "(none supplied)"
    findings = [f"UNSUPPORTED: {c}" for c in state.get("evaluation_unsupported_claims", [])] \
        + list(state.get("evaluation_issues", []))

    result = ask(RevisedLetter, REVISE_LOOP_SYSTEM,
                 f"JOB ADVERTISEMENT:\n{state['job_ad']}\n\n"
                 f"CV:\n{state['cv']}\n\n"
                 f"CANDIDATE'S PAST COVER LETTERS:\n{letters}\n\n"
                 f"EVALUATOR'S FINDINGS:\n" + ("\n".join(f"- {f}" for f in findings) or "(none)") + "\n\n"
                 f"LETTER:\n---\n{state['final_letter']}\n---")

    print(f"[revise] -> {len(result.letter.split())} words")
    for change in result.changes:
        print(f"    - {change}")

    path = save_letter(state["company"], state["role"], result.letter)
    print(f"[revise] -> {path}")

    return {"final_letter": result.letter,
            "changes": state.get("changes", []) + result.changes,
            "revision_count": state.get("revision_count", 0) + 1}


def route_after_evaluate(state: State) -> str:
    """One AUTOMATIC revise pass at most - the same 'one corrective pass, not a loop' rule
    assemble() already follows, so this can't become the two-gates-arguing failure mode the V1
    experiment hit. Once the score passes or that one pass is spent, control goes to
    human_review rather than straight to END - a human gets final say instead of just the
    threshold, and can still request further revisions themselves from there."""
    if state.get("evaluation_score", 5) >= cfg.eval_score_threshold or state.get("revision_count", 0) >= 1:
        return "human_review"
    return "revise"


def human_review(state: State) -> dict:
    """Pauses for a human decision on the finished letter via interrupt() - only when the caller
    set human_in_the_loop (see State). Resume with Command(resume=...) where the payload is
    {"action": "approve"} | {"action": "edit", "letter": "..."} | {"action": "revise", "notes":
    "..." (optional)}. A human-requested revise isn't bounded the way the automatic one is: it
    routes to revise() same as the automatic pass, which loops back to evaluate() and then here
    again - by then revision_count is >= 1, so route_after_evaluate always lands back on
    human_review rather than auto-revising a second time."""
    if not state.get("human_in_the_loop"):
        return {"human_action": "approve"}

    print("[human_review] waiting for a decision...")
    decision = interrupt({
        "gate": "human_review",
        "letter": state["final_letter"],
        "score": state.get("evaluation_score"),
        "issues": state.get("evaluation_issues", []),
        "unsupported_claims": state.get("evaluation_unsupported_claims", []),
    }) or {}

    action = decision.get("action", "approve")
    print(f"[human_review] -> {action}")

    if action == "edit":
        return {"final_letter": decision["letter"], "human_action": "edit"}
    if action == "revise":
        issues = list(state.get("evaluation_issues", []))
        if decision.get("notes"):
            issues = issues + [f"HUMAN NOTE: {decision['notes']}"]
        return {"evaluation_issues": issues, "human_action": "revise"}
    return {"human_action": "approve"}


def route_after_human_review(state: State) -> str:
    return "revise" if state.get("human_action") == "revise" else END


print("assemble, evaluate, revise and human_review defined")


---
## 11. Assemble the graph

Not linear: `research` and `prepare` fan out from `load`, and `para_intro`/`para_body` fan out
further and back in at `assemble`, which only runs once both paragraphs are done. No loops - the
review step (and the closing it writes) lives inside `assemble` and runs exactly once.


In [ ]:
def load(state: State) -> dict:
    """Studio invokes the graph with `{}` - this reads inputs/ from disk itself in that case,
    same as the `load_inputs()` call above. Skipped when the caller already supplied job_ad/cv."""
    if state.get("job_ad") and state.get("cv"):
        return {}
    return load_inputs()


builder = StateGraph(State)

for name, fn in [("load", load), ("prepare", prepare), ("research", research),
                 ("select_reasons", select_reasons), ("para_intro", para_intro),
                 ("qualify", qualify), ("select_qualifications", select_qualifications),
                 ("para_body", para_body),
                 ("assemble", assemble),
                 ("evaluate", evaluate), ("revise", revise), ("human_review", human_review)]:
    builder.add_node(name, fn)

builder.add_edge(START, "load")

# prepare, research and qualify all only need load's output, not each other's - they run
# concurrently. qualify reads job_ad/cv/past_letters directly (not prepare's closings), so it
# gets its own edge from load rather than sitting downstream of prepare - that lets its LLM call
# overlap with research's slow web searches from the start, instead of only starting once
# research's shared tick with prepare has already cleared
builder.add_edge("load", "prepare")
builder.add_edge("load", "research")
builder.add_edge("load", "qualify")

# para_intro and para_body are written independently of each other - each only depends on its own
# upstream branch, so both run concurrently. The closing is no longer a third parallel branch: it's
# written by assemble() itself (see REVISE_SYSTEM), since assemble already needs to read both
# paragraphs to merge them - writing the closing there for free avoids a dedicated LLM call for it
builder.add_edge("research", "select_reasons")
builder.add_edge("select_reasons", "para_intro")
builder.add_edge("qualify", "select_qualifications")
builder.add_edge("select_qualifications", "para_body")

# assemble is the join: a *list* of start nodes in one add_edge call is what actually makes
# LangGraph wait for ALL of them - separate single-source add_edge calls use OR semantics under
# the hood (assemble becomes eligible as soon as any one finishes), which races the others.
# prepare is included since assemble reads its closings (see REVISE_SYSTEM) - harmless for timing,
# since prepare (no LLM call) always finishes long before para_intro/para_body do, but without an
# edge here prepare would dangle straight to END with no visible link to where its output is used
builder.add_edge(["para_intro", "para_body", "prepare"], "assemble")

builder.add_edge("assemble", "evaluate")
builder.add_conditional_edges("evaluate", route_after_evaluate,
                               {"revise": "revise", "human_review": "human_review"})
builder.add_edge("revise", "evaluate")
builder.add_conditional_edges("human_review", route_after_human_review, {"revise": "revise", END: END})

# a checkpointer is required for the interrupt()/Command(resume=...) gates to work at all -
# InMemorySaver is fine here since a thread only needs to survive one process's lifetime (this
# notebook kernel, or one `langgraph dev` server run); nothing here needs it to outlive that
graph = builder.compile(checkpointer=InMemorySaver())
print("graph compiled")

In [ ]:
from IPython.display import Image

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:                     # the PNG comes from a remote renderer
    print(f"(diagram unavailable: {exc})\n")
    print(graph.get_graph().draw_mermaid())

---
## 12. Run it

Around seven model calls, roughly three minutes.


In [ ]:
def prompt_select_reasons(payload: dict) -> dict:
    """Render select_reasons's interrupt payload and collect a selection from stdin."""
    reasons = payload.get("reasons", [])
    print("\n" + "=" * 70)
    print("SELECT REASONS - pick up to 3 for the opening paragraph to build on")
    for i, r in enumerate(reasons):
        print(f"  [{i}] {r}")
    print("-" * 70)

    raw = input("Numbers, comma-separated (blank for none): ").strip()
    selected = [int(x) for x in raw.split(",") if x.strip().isdigit()][:3] if raw else []

    print("Add your own reasons too, if you like - one per line, blank line to finish:")
    custom = []
    while (line := input()) != "":
        custom.append(line)

    return {"selected": selected, "custom": custom}


def prompt_select_qualifications(payload: dict) -> dict:
    """Render select_qualifications's interrupt payload and collect a selection from stdin."""
    qualifications = payload.get("qualifications", [])
    print("\n" + "=" * 70)
    print("SELECT QUALIFICATIONS - pick as many as you think are strongest")
    for i, q in enumerate(qualifications):
        print(f"  [{i}] [{q['relevance']}/5] {q['qualification']}")
        print(f"       evidence: {q['evidence']}")
        print(f"       value to team: {q['value_to_team']}")
    print("-" * 70)

    raw = input("Numbers, comma-separated (blank for none): ").strip()
    selected = [int(x) for x in raw.split(",") if x.strip().isdigit()] if raw else []

    custom = []
    print("Add your own qualifications too, if you like - blank qualification to stop.")
    while True:
        qualification = input("Qualification (blank to stop): ").strip()
        if not qualification:
            break
        value_to_team = input("What would this be worth to this team? ").strip()
        custom.append({"qualification": qualification, "value_to_team": value_to_team})

    return {"selected": selected, "custom": custom}


def prompt_human_review(payload: dict) -> dict:
    """Render human_review's interrupt payload and collect a decision from stdin - Jupyter
    supports blocking input() fine, it just pauses this cell until you answer."""
    print("\n" + "=" * 70)
    print(f"HUMAN REVIEW - score {payload.get('score')}/5")
    for issue in payload.get("issues", []):
        print(f"  issue: {issue}")
    for claim in payload.get("unsupported_claims", []):
        print(f"  unsupported: {claim}")
    print("-" * 70)
    print(payload.get("letter", ""))
    print("-" * 70)

    action = input("approve / edit / revise? [approve] ").strip().lower() or "approve"
    if action == "edit":
        print("Paste the replacement letter, then an empty line to finish:")
        lines = []
        while (line := input()) != "":
            lines.append(line)
        return {"action": "edit", "letter": "\n".join(lines)}
    if action == "revise":
        notes = input("Notes for the reviser (optional): ").strip()
        return {"action": "revise", "notes": notes} if notes else {"action": "revise"}
    return {"action": "approve"}


PROMPT_BY_GATE = {
    "select_reasons": prompt_select_reasons,
    "select_qualifications": prompt_select_qualifications,
    "human_review": prompt_human_review,
}


def invoke_via_studio(payload: dict) -> dict:
    """Run through the `langgraph dev` server so this run shows up as a thread in Studio,
    instead of invoking `graph` in-process (which the server never sees). Falls back to the
    in-process graph when the server isn't running - keeps run_headless.py working headless.

    Either way, every call gets its own thread_id - required by the checkpointer even for runs
    that never hit an interrupt - each of select_reasons, select_qualifications and human_review
    pauses only when its own flag is set (see State), independently of the other two. More than
    one can be pending at once now that qualify runs off load directly (see the graph-wiring
    comment) - select_reasons and select_qualifications can land in the same tick, in which case
    this prompts for all of them before resuming, keyed by each one's own interrupt id (required
    once there's more than one pending; harmless to always do). If this goes through Studio, each
    pause is Studio's own built-in interrupt UI - resume the thread there. The in-process fallback
    instead resumes right here via stdin prompts."""
    config = {"recursion_limit": 30, "configurable": {"thread_id": str(uuid.uuid4())}}

    try:
        from langgraph_sdk import get_sync_client
    except ImportError:
        get_sync_client = None

    if get_sync_client is not None:
        client = get_sync_client(url="http://127.0.0.1:2024")
        try:
            thread = client.threads.create(graph_id="cover_letter")
        except Exception as exc:
            print(f"[studio] dev server not reachable ({type(exc).__name__}); running in-process")
            client = None

        if client is not None:
            print(f"[studio] thread {thread['thread_id']} - open Studio to watch it run")
            return client.runs.wait(thread["thread_id"], "cover_letter",
                                     input=payload, config={"recursion_limit": 30})

    result = graph.invoke(payload, config)
    while "__interrupt__" in result:
        resume_payload = {i.id: PROMPT_BY_GATE[i.value["gate"]](i.value)
                           for i in result["__interrupt__"]}
        result = graph.invoke(Command(resume=resume_payload), config)
    return result


result = invoke_via_studio({**inputs, "select_reasons_in_the_loop": True,
                            "select_qualifications_in_the_loop": True, "human_in_the_loop": True})
print("\n" + "=" * 70)
if "final_letter" in result:
    display(Markdown(result["final_letter"]))
else:
    print("Run paused in Studio - open Studio to review and resume the thread.")

---
## 13. Inspect the pieces

The value of building it paragraph by paragraph is that you can look at each one, and at the
reasoning behind it, rather than at a finished letter you either like or do not.


In [ ]:
print("=" * 70); print("RESEARCH NOTES AND CANDIDATE MOTIVATIONS")
print(result["research_notes"])

print("\n" + "=" * 70); print("QUALIFICATIONS -> VALUE TO THE TEAM")
for item in result["qualifications"]:
    print(f"\n  [{item['relevance']}/5] {item['qualification']}")
    print(f"       evidence : {item['evidence'][:110]}")
    print(f"       value    : {item['value_to_team'][:110]}")

print("\n" + "=" * 70); print("PARAGRAPHS AS WRITTEN")
for label, key in [("INTRO", "para_intro"), ("BODY", "para_body")]:
    print(f"\n--- {label} ({len(result[key].split())} words)\n{result[key]}")

if result.get("changes"):
    print("\n" + "=" * 70); print("CHANGES MADE BY ASSEMBLE")
    for change in result["changes"]:
        print("  - " + change)

In [ ]:
print(f"SOURCES ({len(result['sources'])})")
for source in result["sources"][:12]:
    print(f"  - {source['title'][:65]}\n    {source['url']}")

---
## Worth changing

**A URL that will not fetch.** Most large job boards render postings with JavaScript or require a
login, so `job_url.txt` fails more often than it works and the loader falls back to
`job_posting.txt`. That fallback is the reliable path; treat the URL as a convenience.

**The corrective pass is single-shot by design.** If `assemble` regularly reports unresolved
problems, the fix is a better `qualify` step rather than a second corrective pass - the problems
almost always trace back to a qualification whose evidence was thin.

**`value_to_team` is the field to tune.** If the letter reads as being about the candidate rather
than about the team, that field has gone generic. Tightening its description in `QUALIFY_SYSTEM`
changes the letter more than editing any of the writing prompts.

**No human gate.** The graph runs start to finish. An `interrupt()` before `assemble` would let
you approve the paragraphs before they are stitched.

**Nothing is cached.** Each run re-analyses the CV. Keying `prepare` on a hash of the CV text
would save a call per application.

**Every rendered prompt is written to `outputs/.prompts/`** while `dump_prompts` is on - one file
per node, containing the exact system and user messages that were sent. When a paragraph comes
out wrong, read the prompt that produced it before editing the prompt template.

---
## 14. LangGraph Studio - stepping through a run node by node

This notebook stays the source of truth - prompts, nodes, the graph, all defined above.
[`graph.py`](graph.py) is a thin loader: it reads this notebook's own code cells and executes
them (the same trick [`run_headless.py`](run_headless.py) uses), then exposes the resulting
`graph`. [`langgraph.json`](langgraph.json) points Studio at `./graph.py:graph`.

```bash
langgraph dev
```

This prints a Studio URL (`https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024`) -
open it, logged into your LangSmith account. From there you can:

- invoke the graph with `{}` as input - the `load` node added in section 12 reads `inputs/` from
  disk itself, so there is nothing to paste in
- watch each node run, edit state, jump into a node, resume from any node
- see the same run land in LangSmith as a trace, once `LANGSMITH_TRACING=true` and
  `LANGSMITH_API_KEY` are set in `.env` (added in Setup above)

Edit prompts and nodes here, as usual, and save. `langgraph dev` hot-reloads `graph.py` on file
change, which re-reads this notebook from disk - so a saved edit shows up in Studio without
restarting the server. (Jupyter's autosave can lag a manual save by a few seconds; save explicitly
with Cmd/Ctrl+S if a change isn't picked up.)